In [1]:
import pandas as pd
import os

folder = r"C:\Users\itsma\Desktop\sepsiswatch\data\raw"
all_patients = []

for file in os.listdir(folder):
    if file.endswith('.psv'):
        df = pd.read_csv(os.path.join(folder, file), sep='|')
        df['patient_id'] = file.replace('.psv', '')
        all_patients.append(df)

final_data = pd.concat(all_patients, ignore_index=True)
print(f"Loaded {final_data['patient_id'].nunique():,} patients")

Loaded 36,111 patients


In [2]:
# FUNNEL Analysis

#Stage 1 : All ICU patients
patient_total = final_data['patient_id'].nunique()

# Stage2 : Patient staying more than 6 hours
stayed6 = final_data.groupby('patient_id')['ICULOS'].max()
at_risk = stayed6[stayed6 >= 6].index
count_at_risk = len(at_risk)

#Stage3 : Patient showing early warning signs (HR > 100 or Resp > 22 or MAP < 70)
warning_pat = final_data[(final_data['HR'] > 100) |
                         (final_data['Resp'] > 22) |
                         (final_data['MAP'] < 70)]['patient_id'].unique()
warning_count = len(warning_pat)

#Stage4 : Patient who developed sepsis
sepsis_count = final_data[final_data['SepsisLabel'] == 1]['patient_id'].nunique()

#Stage5 : Sepsis detected late. i.e. after 24 hours in ICU
late_detect =[]
for pid in final_data[final_data['SepsisLabel'] == 1]['patient_id'].unique():
    patient = final_data[final_data['patient_id'] == pid]
    sepsis_hour = patient[patient['SepsisLabel'] == 1]['ICULOS'].min()
    if sepsis_hour > 24:
        late_detect.append(pid)
late_count = len(late_detect)

print("=== PATIENT FUNNEL ===")
print(f"Stage 1 - Total ICU admissions:                 {patient_total:,}")
print(f"Stage 2 - At risk (6+ hours in ICU):            {count_at_risk:,}")
print(f"Stage 3 - Showed warning signs:                 {warning_count:,}")
print(f"Stage 4 - Developed sepsis:                     {sepsis_count:,}")
print(f"Stage 5 - Late detected( 24 hours after):       {late_count:,}")


=== PATIENT FUNNEL ===
Stage 1 - Total ICU admissions:                 36,111
Stage 2 - At risk (6+ hours in ICU):            36,111
Stage 3 - Showed warning signs:                 33,245
Stage 4 - Developed sepsis:                     2,592
Stage 5 - Late detected( 24 hours after):       1,396


In [3]:
import plotly.graph_objects as go

stages = ['ICU Admissions','At Risk (6+ hrs)', 'Showed Warning Signs',
          'Developed Sepsis','Late Detected (24+ hrs)']

values = [36111, 36111, 33245, 2592, 1396]

fig = go.Figure(go.Funnel(
    y = stages,
    x = values,
    textinfo = "value+percent initial",
    marker = dict(color = ['#2196F3','#42A5F5','#FF9800','#F44336',
                           '#B71C1C'])
))

fig.update_layout(title = 'SepsisWatch - Patient Risk Funnel',
                  font = dict(size = 14),
                  height = 500
)

fig.show()